<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_core_variant_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ball — Ball Trap Brain

A toy "ball trap" symbolic model: input characters are tokenized against an alphabet, turned into a ball trajectory through a ring of deflectors, and the deflectors the ball falls through ("holes") produce the output characters.

This notebook contains:
1. The tokenizer and the `Ball_Trap_Brain` model itself
2. Fitness evaluation utilities
3. A genetic algorithm (`main`) that evolves a single brain
4. A `Council_Brain` that combines several independently-trained brains by majority vote
5. A **continuous relaxation** training method (`Ball_Trap_Brain.train_continuous`) that trains a brain with gradient descent instead of the genetic algorithm, then writes the trained parameters straight back into the same instance
6. An optional, guarded cell that reproduces the original end-to-end council training run


## Imports & tokenizer

In [ ]:
import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

## `Ball_Trap_Brain`

Includes the original genetic-algorithm-oriented methods (`clone`, `Mutate`, `shift_alphabet`, ...) plus `train_continuous`, a gradient-based continuous relaxation trainer (see its docstring for details on the relaxation scheme). `train_continuous` mutates the brain in place and returns the loss history.

**Alignment note:** each predicted trajectory position is matched against the target string **right-aligned** (last predicted position <-> last target character), not left-aligned. Test inputs commonly share an invariant prefix (e.g. every `"1+i"` starts with the same `"1+"`), so left-aligning would repeatedly score that shared, input-invariant trajectory against different targets and the loss couldn't tell examples apart. The trailing characters carry the example-specific information, so aligning from the end gives real, discriminative gradient signal.

In [ ]:
class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        directions = [random.choice([1,-1]) for _ in range(alphabet)]
        return directions

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))


        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):

        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        # Passes the standard cap of 10 and the current ball_energy
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        directions.pop(index)
        position += sum(directions)
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output, self.holes_multiplier)

      # Fast shallow copy of the 10,000 elements
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = 1 if self.alphabet_directions[i] == -1 else -1

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

    def train_continuous(self, test_cases, epochs=300, lr=0.05, batch_size=16,
                          temp_start=10.5, temp_end=0.5, sharpness=0.75,
                          warm_start=True, verbose=True):
        """
        Continuous relaxation training.

        Every discrete choice the brain makes is replaced with a differentiable
        stand-in for the duration of training:
          - each deflector's destination index      -> a softmax over all deflectors
          - each deflector's hole/bounce identity    -> a sigmoid (hole probability)
          - each deflector's output symbol (if hole) -> a softmax over self.output
          - each deflector's bounce value (if not)   -> a free real-valued scalar
          - each alphabet direction (+-1)            -> tanh of a free scalar
          - each alphabet start position              -> a softmax over all deflectors

        The ricochet loop is unrolled for `cap` steps and the hard indexing /
        branching / while-loop of the discrete simulator is replaced by
        probability-weighted ("expected value") versions of the same
        arithmetic, so the whole forward pass is differentiable and can be
        trained with gradient descent against the task loss instead of the
        genetic algorithm's mutate-and-select loop. A temperature is annealed
        from `temp_start` down to `temp_end` over training so the softmax
        distributions sharpen towards one-hot (i.e. towards the original
        discrete model) as training progresses.

        Because the discrete model can emit a variable number of output
        symbols per input, this relaxation makes one simplifying assumption:
        it treats the *first hole reached* by each character's trajectory as
        that character's contribution to the output, and aligns those
        per-character predictions against the target string position by
        position (extra predicted or target positions beyond the shorter of
        the two are simply not scored). This keeps the forward pass a fixed,
        known length so it can be trained with ordinary cross-entropy.

        After training, every continuous parameter is discretized (argmax /
        sign / rounding) and written directly back into this instance's own
        `deflectors_list`, `alphabet_directions` and `alphabet_start_positions`
        attributes -- no separate trained object is kept around, and every
        other method on this class (compute_input, ricochet, clone, Mutate,
        display_deflectors, ...) keeps working exactly as before, just with
        gradient-trained values in place of the old ones.

        Returns the per-epoch loss history (list of floats).
        """
        import torch
        import torch.nn.functional as F

        D = len(self.deflectors_list)
        A = len(self.alphabet)
        K = len(self.output)
        cap = 10
        ball_energy = float(self.ball_energy)
        output_index = {sym: k for k, sym in enumerate(self.output)}
        positions = torch.arange(D, dtype=torch.float32)
        abs_diff = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # abs_diff[d, d'] = |d - d'|

        CONF = 4.0  # logit magnitude used so a warm-started softmax closely matches the current discrete choice
        dest_logits = torch.zeros(D, D)
        hole_logit = torch.zeros(D)
        symbol_logits = torch.zeros(D, K)
        bounce_val = torch.zeros(D)
        direction_param = torch.zeros(A)
        start_logits = torch.zeros(A, D)

        if warm_start:
            for i, (dest, val) in enumerate(self.deflectors_list):
                dest_logits[i, dest] = CONF
                if isinstance(val, int):
                    hole_logit[i] = -CONF
                    bounce_val[i] = float(val)
                else:
                    hole_logit[i] = CONF
                    symbol_logits[i, output_index[val]] = CONF
            for i, d in enumerate(self.alphabet_directions):
                direction_param[i] = CONF if d == 1 else -CONF
            for i, p in enumerate(self.alphabet_start_positions):
                start_logits[i, p] = CONF
        else:
            dest_logits.normal_(0, 0.1)
            hole_logit.normal_(0, 0.1)
            symbol_logits.normal_(0, 0.1)
            bounce_val.normal_(0, 1.0)
            direction_param.normal_(0, 0.1)
            start_logits.normal_(0, 0.1)

        for p in (dest_logits, hole_logit, symbol_logits, bounce_val, direction_param, start_logits):
            p.requires_grad_(True)

        optimizer = torch.optim.Adam(
            [dest_logits, hole_logit, symbol_logits, bounce_val, direction_param, start_logits], lr=lr
        )

        def soft_index_dist(travel_value, temp):
            # Differentiable stand-in for `t % n` followed by hard indexing: a distribution
            # over deflector indices peaked (with width set by temperature) at travel_value.
            diff = positions - travel_value
            diff = torch.remainder(diff + D / 2, D) - D / 2
            logits = -(diff ** 2) / (sharpness * temp)
            return F.softmax(logits, dim=0)

        def simulate(input_indices, temp):
            L = len(input_indices)
            directions_soft = torch.tanh(direction_param)              # (A,) relaxation of +-1
            dirs = directions_soft[input_indices]                      # (L,)
            total_dir = dirs.sum()

            start_probs_all = F.softmax(start_logits / temp, dim=1)    # (A, D)
            expected_start_all = (start_probs_all * positions).sum(dim=1)  # (A,)
            starts = expected_start_all[input_indices]                 # (L,)

            dest_probs = F.softmax(dest_logits / temp, dim=1)          # (D, D)
            hole_p = torch.sigmoid(hole_logit)                         # (D,)
            symbol_probs = F.softmax(symbol_logits / temp, dim=1)      # (D, K)
            weighted_abs = (dest_probs * abs_diff).sum(dim=1)          # (D,) expected hop distance

            outputs = []
            for i in range(L):
                travel_i = starts[i] + (total_dir - dirs[i])           # matches Brain.travel()
                state = soft_index_dist(travel_i, temp)                # distribution over current deflector
                be = torch.tensor(ball_energy)
                alive = torch.tensor(1.0)          # soft "be > 0 and cap_idx < cap" gate
                not_hole_yet = torch.tensor(1.0)   # soft "no hole reached yet" gate
                out_dist = torch.zeros(K)

                for _ in range(cap):
                    p_hole = (state * hole_p).sum()
                    p_bounce_energy = (state * (1 - hole_p) * bounce_val).sum()
                    exp_dist = (state * weighted_abs).sum()

                    p_first_hole = alive * not_hole_yet * p_hole
                    sym_dist = (state.unsqueeze(1) * hole_p.unsqueeze(1) * symbol_probs).sum(dim=0)
                    sym_dist = sym_dist / (p_hole + 1e-8)
                    out_dist = out_dist + p_first_hole * sym_dist

                    energy_delta = p_hole * 1.0 + p_bounce_energy - exp_dist
                    be = be + energy_delta
                    alive = alive * torch.sigmoid(be * 2.0)
                    not_hole_yet = not_hole_yet * (1 - p_hole)

                    state = state @ dest_probs

                outputs.append(out_dist)
            return outputs

        history = []
        for epoch in range(epochs):
            # frac = epoch / max(1, epochs - 1)
            # temp = temp_start + (temp_end - temp_start) * frac
            decay_rate = 0.985   # temp *= decay_rate each epoch
            temp = temp_start * (decay_rate ** epoch)
            temp = max(temp, temp_end)
            batch = random.sample(test_cases, min(batch_size, len(test_cases)))

            optimizer.zero_grad()
            total_loss = torch.tensor(0.0)
            count = 0
            for inp, target in batch:
                tokens = TokenizeInput(inp, self.alphabet)
                indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
                if not indices:
                    continue
                outs = simulate(indices, temp)
                target_indices = [output_index[c] for c in target if c in output_index]
                n = min(len(outs), len(target_indices))
                # Right-align: match the *last* n predicted-trajectory positions against the
                # *last* n target characters. Test inputs typically share a literal, invariant
                # prefix (e.g. every "1+i" starts with the same "1+"), so left-truncating to the
                # first n positions would almost always score the same input-invariant trajectory
                # against different targets -- gradients then can't distinguish examples at all.
                # The trailing characters of the input are where the example-specific information
                # (e.g. the digits of i) actually lives, so align from the end instead.
                out_offset = len(outs) - n
                tgt_offset = len(target_indices) - n
                for i in range(n):
                    pred = outs[out_offset + i]
                    pred_norm = pred / (pred.sum() + 1e-8)
                    total_loss = total_loss - torch.log(pred_norm[target_indices[tgt_offset + i]] + 1e-8)
                    count += 1

            if count == 0:
                continue
            total_loss = total_loss / count
            total_loss.backward()
            optimizer.step()
            history.append(total_loss.item())

            if verbose and (epoch % max(1, epochs // 20) == 0 or epoch == epochs - 1):
                print(f"[continuous] epoch {epoch}: loss={total_loss.item()} temp={temp}")

        # Discretize the trained continuous parameters and write them back into
        # this same Ball_Trap_Brain instance's own attributes.
        with torch.no_grad():
            for i in range(D):
                dest = int(torch.argmax(dest_logits[i]).item())
                if torch.sigmoid(hole_logit[i]).item() > 0.5:
                    val = self.output[int(torch.argmax(symbol_logits[i]).item())]
                else:
                    val = int(round(bounce_val[i].item()))
                self.deflectors_list[i][0] = dest
                self.deflectors_list[i][1] = val

            for i in range(A):
                self.alphabet_directions[i] = 1 if direction_param[i].item() > 0 else -1
                self.alphabet_start_positions[i] = int(torch.argmax(start_logits[i]).item())

        return history

## Test case generation

In [ ]:
import random
import string
def generate_test_cases(amount, number, indiv=True):

  test_cases = []
  if indiv == True:
    test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
  else:
    test_cases = [(f'1+{i}', f'{1+i}') for i in range(amount)]

  return test_cases

## Fitness evaluation

In [ ]:
"""
Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.
"""

import difflib
import math


def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0


import concurrent.futures

def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

def evaluate_population_parallel(population, test_cases, ratio=False):
    # Package arguments for the worker pool
    args = [(ind, test_cases, ratio) for ind in population]

    # Run across available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain

## Genetic algorithm (`main`)

In [ ]:
"""
Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.
"""

string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

import copy
import difflib
import random
import shelve

# Test cases: pairs of (input_string, expected_output_string) used for fitness evaluation


alphabet = [f'{i}' for i in range(50)] + ['+'] #+ list("abcdefghijklmnopqrstuvwxyz")  # Define the alphabet used in the brains
output = [f'{i}' for i in range(10)]

def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    # Evolution hyperparameters
    population_size = pop_size

    # Initialize population with random brains
    # Parameters: 10 deflectors, 2 bounce energy, alphabet of 6 characters

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    #a = [mini_batch_0.pop(0) for i in range(10)]
    highest_score = -1
    # Run evolution loop
    for generation in range(generations):

        # Evaluate all brains and find the best performer
        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)



        # Check for perfect solution
        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual
          #for i in range(10):
            #a.append(mini_batch_0.pop(0))


        # Create next generation by mutating the best brain multiple times
        # New code: passes best_fitness, individual_probability, base_probability
        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

training_1 = generate_test_cases(amount=100, number='1', indiv=True)

## `Council_Brain` & council training

In [ ]:
from collections import Counter

class Council_Brain:
    def __init__(self, brains):
        """
        Initializes the council with a list of separately trained brains.
        """
        self.brains = brains

    def compute_input(self, input_val):
        """
        Each brain processes the input separately.
        The council votes on the final answer by finding the most common output.
        """
        votes = []
        for brain in self.brains:
            # Each brain outputs a list of characters; join them to form the string
            output_list = brain.compute_input(input_val)
            output_str = ''.join(output_list)
            votes.append(output_str)

        # Handle empty cases securely
        if not votes:
            return []

        # Tally the votes and find the majority
        vote_counts = Counter(votes)
        majority_vote = vote_counts.most_common(1)[0][0]

        # Return as a list of characters to perfectly match the original Ball_Trap_Brain format
        return list(majority_vote)

def train_council(num_brains, training_data, alphabet, output_format, generations=50, pop_size=100):
    """
    Trains a council where each brain is trained separately using the genetic algorithm.
    """
    trained_brains = []

    for i in range(num_brains):
        print(f"\n=====================================")
        print(f" Training Council Brain {i+1} / {num_brains}")
        print(f"=====================================")
        base_brain = Ball_Trap_Brain(100, 10, alphabet, output, 2)
        # Initialize a fresh brain for this specific council member
        #base_brain = Ball_Trap_Brain(100, 10, alphabet, output_format, 2)

        # Train separately using the existing main GA loop
        for j in range(1):
          print(f'---{j}---')
          base_brain = main(
              mini_batch=random.sample(training_data, 10),
              alphabet=alphabet,
              generations=generations,
              pop_size=pop_size,
              brain=base_brain,
              ratio=True
          )
        trained_brains.append(base_brain)

    return Council_Brain(trained_brains)

## Continuous relaxation training — demo

Trains a brain with gradient descent (requires `torch`; installed via `pip install torch` if missing) instead of the genetic algorithm, then shows that the resulting parameters have been written back into the same `Ball_Trap_Brain` instance and that the instance is fully usable afterward. Note how predictions differ across inputs after training -- with the right-aligned loss, different examples actually produce different gradient signal.

In [ ]:
# !pip install torch --quiet   # uncomment if torch isn't already installed

import random

demo_brain = Ball_Trap_Brain(100, 4, alphabet, output, 2)
demo_cases = generate_test_cases(amount=10, number='1', indiv=False)
random.shuffle(demo_cases)  # '1+i' -> str(1+i)

print('Before:', demo_brain.deflectors_list[:5])

history = demo_brain.train_continuous(
    demo_cases,
    epochs=100,
    lr=2,
    batch_size=10,
    warm_start=True,
    verbose=True,
    temp_start=30,
    temp_end=0.01,
)

print('After: ', demo_brain.deflectors_list[:5])
print('loss[0] ->', history[0], ' loss[-1] ->', history[-1])

for inp, ans in demo_cases[:10]:
    print(inp, '->', ''.join(demo_brain.compute_input(inp)), ' (target', ans, ')')

fitness = sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_cases) / len(demo_cases)
print('ratio fitness after continuous training:', fitness)

Before: [[34, 4], [13, 4], [24, 4], [16, 4], [90, 4]]
[continuous] epoch 0: loss=2.304115056991577 temp=30.0
[continuous] epoch 5: loss=2.1123464107513428 temp=27.81649507096875
[continuous] epoch 10: loss=1.7364776134490967 temp=25.79191326777429
[continuous] epoch 15: loss=1.4528149366378784 temp=23.914687609463233
[continuous] epoch 20: loss=1.2939525842666626 temp=22.174093000413045
[continuous] epoch 25: loss=1.225352168083191 temp=20.56018495497307
[continuous] epoch 30: loss=1.1762158870697021 temp=19.063742781940473
[continuous] epoch 35: loss=1.1098151206970215 temp=17.676216904268774
[continuous] epoch 40: loss=1.051438808441162 temp=16.389680013032226
[continuous] epoch 45: loss=1.0044487714767456 temp=15.196781776575534
[continuous] epoch 50: loss=0.9733139276504517 temp=14.090706846090033
[continuous] epoch 55: loss=0.9540042877197266 temp=13.065135917690966
[continuous] epoch 60: loss=0.9353542327880859 temp=12.114209628532917
[continuous] epoch 65: loss=0.916159987449646

## Full council training run (optional, slow)

This reproduces the original script's `if __name__ == '__main__':` block: it trains a 50-brain council with the genetic algorithm (400 generations each) and evaluates it. This can take a long time, so it's guarded behind `RUN_FULL_TRAINING`.

In [ ]:
# RUN_FULL_TRAINING = False  # set to True to actually run the full council training

# if RUN_FULL_TRAINING:
#     # Generate the test cases using the existing generation function
#     test_cases = generate_test_cases(amount=1000, number='1', indiv=False)
#     random.shuffle(test_cases)

#     # Split the dataset
#     training_0 = test_cases[:int(0.5 * len(test_cases))]
#     training_test = test_cases[int(0.5 * len(test_cases)):]


#     # Define council parameters
#     COUNCIL_SIZE = 50      # Number of voting brains in the council
#     GENERATIONS = 400     # Generations per brain
#     POP_SIZE = 100        # Population size per generation

#     # Train the council (each brain is trained entirely separately)
#     print(f"Initializing Council Training with {COUNCIL_SIZE} brains...")
#     council = train_council(
#         num_brains=COUNCIL_SIZE,
#         training_data=training_0,
#         alphabet=alphabet,
#         output_format=output,
#         generations=GENERATIONS,
#         pop_size=POP_SIZE
#     )



#     # Evaluate the entire council's performance against the testing set
#     # Note: evaluate_population_parallel relies on a list of brains, so we wrap the council in a list
#     print("\nEvaluating Council performance on test batch...")
#     council_fitness, _ = evaluate_population_parallel([council], training_test, False)
#     council_fitness_training, _ = evaluate_population_parallel([council], training_0, False)
#     print(f"Overall Council Fitness: {council_fitness}, Council training fitness: {council_fitness_training}")